# Python practice 6: iterators, generators and itertools

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-practice-06-iterators-and-itertools.ipynb)

14 exercises on iterators, comprehensions, generators and itertools. Each exercise names the [reference entries](../learning/python-reference-06-iterators-and-itertools.ipynb) it practises.

Write each function in the empty cell under its exercise and compare your results with the sample calls. Open the solution cell when you are done or stuck. 8 easy, 5 medium, 1 hard.

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Chunk a stream into batches](#exercise-1-chunk-a-stream-into-batches) | Easy | [Generator functions and `yield`](../learning/python-reference-06-iterators-and-itertools.ipynb#generator-functions-and-yield) |
| 2 | [Flatten arbitrarily nested lists](#exercise-2-flatten-arbitrarily-nested-lists) | Medium | [Generator functions and `yield`](../learning/python-reference-06-iterators-and-itertools.ipynb#generator-functions-and-yield) |
| 3 | [Transpose rows safely](#exercise-3-transpose-rows-safely) | Easy | [Built-in iteration helpers](../learning/python-reference-06-iterators-and-itertools.ipynb#built-in-iteration-helpers) |
| 4 | [Dict comprehension inversion](#exercise-4-dict-comprehension-inversion) | Easy | [List, dict, set and generator comprehensions](../learning/python-reference-06-iterators-and-itertools.ipynb#list-dict-set-and-generator-comprehensions) |
| 5 | [Sessionise click streams](#exercise-5-sessionise-click-streams) | Hard | [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate) |
| 6 | [Changes between consecutive readings](#exercise-6-changes-between-consecutive-readings) | Easy | [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate) |
| 7 | [Run-length encoding](#exercise-7-run-length-encoding) | Medium | [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate) |
| 8 | [Running balance and drawdown](#exercise-8-running-balance-and-drawdown) | Medium | [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate) |
| 9 | [Parameter grid](#exercise-9-parameter-grid) | Medium | [Combinatorics: `product`, `permutations`, `combinations`, `zip_longest`, `starmap`, `tee`](../learning/python-reference-06-iterators-and-itertools.ipynb#combinatorics-product-permutations-combinations-zip_longest-starmap-tee) |
| 10 | [Pairs that sum to a target](#exercise-10-pairs-that-sum-to-a-target) | Easy | [Combinatorics: `product`, `permutations`, `combinations`, `zip_longest`, `starmap`, `tee`](../learning/python-reference-06-iterators-and-itertools.ipynb#combinatorics-product-permutations-combinations-zip_longest-starmap-tee) |
| 11 | [First n matching items lazily](#exercise-11-first-n-matching-items-lazily) | Easy | [Slicing and chaining: `islice`, `chain`, `takewhile`, `dropwhile`](../learning/python-reference-06-iterators-and-itertools.ipynb#slicing-and-chaining-islice-chain-takewhile-dropwhile) |
| 12 | [Skip a file header](#exercise-12-skip-a-file-header) | Easy | [Slicing and chaining: `islice`, `chain`, `takewhile`, `dropwhile`](../learning/python-reference-06-iterators-and-itertools.ipynb#slicing-and-chaining-islice-chain-takewhile-dropwhile) |
| 13 | [Merge columns of unequal length](#exercise-13-merge-columns-of-unequal-length) | Easy | [Combinatorics: `product`, `permutations`, `combinations`, `zip_longest`, `starmap`, `tee`](../learning/python-reference-06-iterators-and-itertools.ipynb#combinatorics-product-permutations-combinations-zip_longest-starmap-tee) |
| 14 | [Top value per group](#exercise-14-top-value-per-group) | Medium | [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate) |

*Last reviewed: November 2026, against Python 3.13.*

In [ ]:
# Imports used by the sample calls and the solutions.
import io
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from fractions import Fraction

## Exercise 1: Chunk a stream into batches

**Difficulty:** Easy · about 5 min · **Practises:** [Generator functions and `yield`](../learning/python-reference-06-iterators-and-itertools.ipynb#generator-functions-and-yield)

*Bulk inserts and API calls send rows in fixed-size batches.*

Write a generator `chunks(items, size)` yielding lists of up to `size` items from any iterable. Raise `ValueError` if `size < 1`. Return `list(chunks(...))` in checks.

**Sample calls**

| call | expected |
| --- | --- |
| `list(chunks(range(6), 3))` | `[[0, 1, 2], [3, 4, 5]]` |
| `list(chunks(range(7), 3))` | `[[0, 1, 2], [3, 4, 5], [6]]` |
| `list(chunks([], 2))` | `[]` |

In [ ]:
def chunks(items, size):
    # your code here
    pass

In [ ]:
from itertools import islice

def chunks(items, size):
    if size < 1:
        raise ValueError("size must be at least 1")
    it = iter(items)
    while batch := list(islice(it, size)):
        yield batch

## Exercise 2: Flatten arbitrarily nested lists

**Difficulty:** Medium · about 10 min · **Practises:** [Generator functions and `yield`](../learning/python-reference-06-iterators-and-itertools.ipynb#generator-functions-and-yield)

*Nested JSON arrays often need flattening before loading.*

Write a generator `flatten(items)` yielding leaves of nested lists and tuples in order; strings and other values are leaves.

**Sample calls**

| call | expected |
| --- | --- |
| `list(flatten([1, [2, [3, (4, 5)]], 6]))` | `[1, 2, 3, 4, 5, 6]` |
| `list(flatten([1, 2]))` | `[1, 2]` |
| `list(flatten([]))` | `[]` |

In [ ]:
def flatten(items):
    # your code here
    pass

In [ ]:
def flatten(items):
    for item in items:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)
        else:
            yield item

## Exercise 3: Transpose rows safely

**Difficulty:** Easy · about 5 min · **Practises:** [Built-in iteration helpers](../learning/python-reference-06-iterators-and-itertools.ipynb#built-in-iteration-helpers)

*Turning rows into columns, without silently dropping data.*

Write `columns(header, rows)` returning a dict column → list of values. Every row must have the header's length; otherwise raise `ValueError` (use `zip(..., strict=True)`).

**Sample calls**

| call | expected |
| --- | --- |
| `columns(["id", "name"], [(1, "a"), (2, "b")])` | `{'id': [1, 2], 'name': ['a', 'b']}` |
| `columns(["id"], [])` | `{'id': []}` |
| `columns(["x", "y"], [(1, 2)])` | `{'x': [1], 'y': [2]}` |

In [ ]:
def columns(header, rows):
    # your code here
    pass

In [ ]:
def columns(header, rows):
    cols = list(zip(*rows, strict=True)) if rows else [()] * len(header)
    return {name: list(values) for name, values in zip(header, cols, strict=True)}

## Exercise 4: Dict comprehension inversion

**Difficulty:** Easy · about 5 min · **Practises:** [List, dict, set and generator comprehensions](../learning/python-reference-06-iterators-and-itertools.ipynb#list-dict-set-and-generator-comprehensions)

*Reverse lookups from code → label maps.*

Write `invert(mapping)` returning value → sorted list of keys that map to it.

**Sample calls**

| call | expected |
| --- | --- |
| `invert({"a": 1, "b": 2})` | `{1: ['a'], 2: ['b']}` |
| `invert({"b": 1, "a": 1, "c": 2})` | `{1: ['a', 'b'], 2: ['c']}` |
| `invert({})` | `{}` |

In [ ]:
def invert(mapping):
    # your code here
    pass

In [ ]:
def invert(mapping):
    values = {v for v in mapping.values()}
    return {v: sorted(k for k, val in mapping.items() if val == v) for v in values}

## Exercise 5: Sessionise click streams

**Difficulty:** Hard · about 20 min · **Practises:** [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate)

*Sessionisation (a new session after an idle gap) is a common analytics interview question.*

Write `sessions(timestamps, gap)` taking unsorted integer timestamps for one user and returning a list of `(start, end, count)` per session, where a new session starts when the difference to the previous timestamp is greater than `gap`.

**Sample calls**

| call | expected |
| --- | --- |
| `sessions([1, 2, 3, 20, 21], 5)` | `[(1, 3, 3), (20, 21, 2)]` |
| `sessions([21, 1, 20, 3, 2], 5)` | `[(1, 3, 3), (20, 21, 2)]` |
| `sessions([], 5)` | `[]` |

In [ ]:
def sessions(timestamps, gap):
    # your code here
    pass

In [ ]:
from itertools import accumulate, groupby, pairwise

def sessions(timestamps, gap):
    ts = sorted(timestamps)
    if not ts:
        return []
    breaks = [0] + [1 if b - a > gap else 0 for a, b in pairwise(ts)]
    ids = accumulate(breaks)
    out = []
    for _, group in groupby(zip(ids, ts), key=lambda p: p[0]):
        values = [t for _, t in group]
        out.append((values[0], values[-1], len(values)))
    return out

## Exercise 6: Changes between consecutive readings

**Difficulty:** Easy · about 5 min · **Practises:** [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate)

*Deltas between snapshots: counters, balances, row counts per load.*

Write `deltas(values)` returning the differences between consecutive values.

**Sample calls**

| call | expected |
| --- | --- |
| `deltas([10, 13, 13, 20])` | `[3, 0, 7]` |
| `deltas([5])` | `[]` |
| `deltas([])` | `[]` |

In [ ]:
def deltas(values):
    # your code here
    pass

In [ ]:
from itertools import pairwise

def deltas(values):
    return [b - a for a, b in pairwise(values)]

## Exercise 7: Run-length encoding

**Difficulty:** Medium · about 10 min · **Practises:** [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate)

*Compressing repeated values; also a classic string interview question.*

Write `rle(text)` returning a string like `a3b1c2` for `aaabcc`, and `unrle(encoded)` reversing it (counts may have several digits). Checks call both.

**Sample calls**

| call | expected |
| --- | --- |
| `rle("aaabcc")` | `'a3b1c2'` |
| `unrle("a3b1c2")` | `'aaabcc'` |
| `unrle(rle("zzzzzzzzzzzzyx"))` | `'zzzzzzzzzzzzyx'` |

In [ ]:
def rle(text):
    # your code here
    pass


def unrle(encoded):
    # your code here
    pass

In [ ]:
import re
from itertools import groupby

def rle(text):
    return "".join(f"{ch}{len(list(group))}" for ch, group in groupby(text))

def unrle(encoded):
    return "".join(ch * int(n) for ch, n in re.findall(r"(\D)(\d+)", encoded))

## Exercise 8: Running balance and drawdown

**Difficulty:** Medium · about 10 min · **Practises:** [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate)

*Running totals and running maxima drive balance and drawdown reports.*

Write `drawdown(changes, start=0)` returning `(balances, max_drawdown)` where balances are the running balance after each change and max_drawdown is the largest drop from a running peak (peaks include `start`).

**Sample calls**

| call | expected |
| --- | --- |
| `drawdown([5, -3, 4, -8, 2])` | `([5, 2, 6, -2, 0], 8)` |
| `drawdown([1, 1])` | `([1, 2], 0)` |
| `drawdown([])` | `([], 0)` |

In [ ]:
def drawdown(changes, start=0):
    # your code here
    pass

In [ ]:
from itertools import accumulate

def drawdown(changes, start=0):
    balances = list(accumulate(changes, initial=start))[1:]
    peaks = accumulate([start] + balances, max)
    worst = max((peak - bal for peak, bal in zip(peaks, [start] + balances)), default=0)
    return balances, worst

## Exercise 9: Parameter grid

**Difficulty:** Medium · about 10 min · **Practises:** [Combinatorics: `product`, `permutations`, `combinations`, `zip_longest`, `starmap`, `tee`](../learning/python-reference-06-iterators-and-itertools.ipynb#combinatorics-product-permutations-combinations-zip_longest-starmap-tee)

*Expanding a config grid into job runs (environments × regions, or hyperparameters).*

Write `grid(params)` taking a dict of name → list of values and returning a list of dicts, one per combination, in the order `itertools.product` produces over the dict's key order.

**Sample calls**

| call | expected |
| --- | --- |
| `grid({"env": ["dev", "prod"], "region": ["us", "eu"]})` | `[{'env': 'dev', 'region': 'us'}, {'env': 'dev', 'region': 'eu'}, {'env': 'prod', 'region': 'us'}, {'env': 'prod', 'region': 'eu'}]` |
| `grid({"x": [1, 2]})` | `[{'x': 1}, {'x': 2}]` |
| `grid({})` | `[{}]` |

In [ ]:
def grid(params):
    # your code here
    pass

In [ ]:
from itertools import product

def grid(params):
    keys = list(params)
    return [dict(zip(keys, combo)) for combo in product(*(params[k] for k in keys))]

## Exercise 10: Pairs that sum to a target

**Difficulty:** Easy · about 5 min · **Practises:** [Combinatorics: `product`, `permutations`, `combinations`, `zip_longest`, `starmap`, `tee`](../learning/python-reference-06-iterators-and-itertools.ipynb#combinatorics-product-permutations-combinations-zip_longest-starmap-tee)

*Brute force with `combinations` is the baseline to beat in two-sum style questions.*

Write `pairs_summing(nums, target)` returning the index pairs `(i, j)` with `i < j` and `nums[i] + nums[j] == target`, in lexicographic order.

**Sample calls**

| call | expected |
| --- | --- |
| `pairs_summing([2, 7, 11, 15], 9)` | `[(0, 1)]` |
| `pairs_summing([1, 2, 3, 4, 5], 6)` | `[(0, 4), (1, 3)]` |
| `pairs_summing([1, 2], 9)` | `[]` |

In [ ]:
def pairs_summing(nums, target):
    # your code here
    pass

In [ ]:
from itertools import combinations

def pairs_summing(nums, target):
    return [(i, j) for (i, a), (j, b) in combinations(enumerate(nums), 2) if a + b == target]

## Exercise 11: First n matching items lazily

**Difficulty:** Easy · about 5 min · **Practises:** [Slicing and chaining: `islice`, `chain`, `takewhile`, `dropwhile`](../learning/python-reference-06-iterators-and-itertools.ipynb#slicing-and-chaining-islice-chain-takewhile-dropwhile)

*Peeking at the first few bad rows of a huge file without reading all of it.*

Write `first_n(items, predicate, n)` returning a list of the first `n` items satisfying `predicate`, consuming no more of `items` than needed.

**Sample calls**

| call | expected |
| --- | --- |
| `first_n(range(100), lambda x: x % 2 == 0, 3)` | `[0, 2, 4]` |
| `first_n([1, 3, 4], lambda x: x % 2 == 0, 5)` | `[4]` |
| `first_n([1], bool, 0)` | `[]` |

In [ ]:
def first_n(items, predicate, n):
    # your code here
    pass

In [ ]:
from itertools import islice

def first_n(items, predicate, n):
    return list(islice(filter(predicate, items), n))

## Exercise 12: Skip a file header

**Difficulty:** Easy · about 5 min · **Practises:** [Slicing and chaining: `islice`, `chain`, `takewhile`, `dropwhile`](../learning/python-reference-06-iterators-and-itertools.ipynb#slicing-and-chaining-islice-chain-takewhile-dropwhile)

*Exports often start with comment lines before the data.*

Write `data_lines(lines)` returning the stripped lines after any leading lines that are blank or start with `#`; later comment lines are kept.

**Sample calls**

| call | expected |
| --- | --- |
| `data_lines(["# export", "", "id,name", "1,a"])` | `['id,name', '1,a']` |
| `data_lines(["a", "b"])` | `['a', 'b']` |
| `data_lines(["#", " "])` | `[]` |

In [ ]:
def data_lines(lines):
    # your code here
    pass

In [ ]:
from itertools import dropwhile

def data_lines(lines):
    is_preamble = lambda line: not line.strip() or line.lstrip().startswith("#")
    return [line.strip() for line in dropwhile(is_preamble, lines)]

## Exercise 13: Merge columns of unequal length

**Difficulty:** Easy · about 5 min · **Practises:** [Combinatorics: `product`, `permutations`, `combinations`, `zip_longest`, `starmap`, `tee`](../learning/python-reference-06-iterators-and-itertools.ipynb#combinatorics-product-permutations-combinations-zip_longest-starmap-tee)

*Lining up lists of different lengths without losing the tail.*

Write `side_by_side(left, right, fill=None)` returning tuples pairing items, padding the shorter list with `fill`.

**Sample calls**

| call | expected |
| --- | --- |
| `side_by_side([1, 2, 3], ["a"])` | `[(1, 'a'), (2, None), (3, None)]` |
| `side_by_side([], [1, 2], fill=0)` | `[(0, 1), (0, 2)]` |
| `side_by_side([1], [2])` | `[(1, 2)]` |

In [ ]:
def side_by_side(left, right, fill=None):
    # your code here
    pass

In [ ]:
from itertools import zip_longest

def side_by_side(left, right, fill=None):
    return list(zip_longest(left, right, fillvalue=fill))

## Exercise 14: Top value per group

**Difficulty:** Medium · about 10 min · **Practises:** [Grouping and windows: `groupby`, `pairwise`, `batched`, `accumulate`](../learning/python-reference-06-iterators-and-itertools.ipynb#grouping-and-windows-groupby-pairwise-batched-accumulate)

*Per-key maxima without pandas, the shape of many groupby interview questions.*

Write `best_per_region(rows)` where rows are `(region, name, amount)`; return a list of `(region, name)` with the highest amount per region (ties: alphabetically first name), regions sorted. Use `sorted` + `groupby`.

**Sample calls**

| call | expected |
| --- | --- |
| `best_per_region([("n", "a", 5), ("s", "b", 9), ("n", "c", 7)])` | `[('n', 'c'), ('s', 'b')]` |
| `best_per_region([("n", "z", 5), ("n", "a", 5)])` | `[('n', 'a')]` |
| `best_per_region([])` | `[]` |

In [ ]:
def best_per_region(rows):
    # your code here
    pass

In [ ]:
from itertools import groupby

def best_per_region(rows):
    ordered = sorted(rows, key=lambda r: (r[0], -r[2], r[1]))
    return [(region, next(group)[1]) for region, group in groupby(ordered, key=lambda r: r[0])]

**Next:** look anything up in [Python reference 6: iterators, generators and itertools](../learning/python-reference-06-iterators-and-itertools.ipynb), or try the [Python challenges](python-challenges-01-arrays-and-hashing.ipynb) and [Python coding problems](python-coding-problems.ipynb).